<img src='https://upload.wikimedia.org/wikipedia/commons/5/58/Uber_logo_2018.svg' style='filter: invert(52%) sepia(100%) saturate(332%) hue-rotate(183deg) brightness(67%) contrast(165%);' alt='UBER LOGO' width='20%'/>

# Uber NYC Hot-Zone Detection

**Objective:** identify where Uber demand concentrates in New York City at different times, 
so drivers can be routed to high-demand areas before riders start waiting too long.

**Approach:** cluster raw 2014 pickup coordinates (KMeans) independently for every 
day-of-week × hour combination to find geographic hot zones, then cross-check the results 
against a second, independent 2015 dataset (zone-based, frequency-ranked) for the same time slot.

# 1. Setup

## 1A. Import libraries

In [1]:
# -----------------------------------------------------------------------------
# IMPORTS
# -----------------------------------------------------------------------------

# Data manipulation
import pandas as pd
import numpy as np

# Data prediction
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import haversine_distances

# Data visualization
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio



pd.options.display.float_format = '{:,.4f}'.format
px.defaults.template = 'plotly_white'

In [2]:
# -----------------------------------------------------------------------------
# DATA LOADING
# -----------------------------------------------------------------------------

path = 'uber-trip-data/'

files_name = ['uber-raw-data-apr14.csv', 'uber-raw-data-may14.csv', 
            'uber-raw-data-jun14.csv', 'uber-raw-data-jul14.csv', 
            'uber-raw-data-aug14.csv', 'uber-raw-data-sep14.csv']
raw_2015 = 'uber-raw-data-janjune-15.csv'
taxi_file = 'taxi-zone-lookup.csv'

uber_dfs = []
for file in files_name:
    file_path = f'{path}{file}'
    uber_data = pd.read_csv(file_path, parse_dates=['Date/Time'])
    uber_dfs.append(uber_data)

uber_df = pd.concat(uber_dfs, ignore_index=True)

In [3]:
uber_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4534327 entries, 0 to 4534326
Data columns (total 4 columns):
 #   Column     Dtype         
---  ------     -----         
 0   Date/Time  datetime64[us]
 1   Lat        float64       
 2   Lon        float64       
 3   Base       str           
dtypes: datetime64[us](1), float64(2), str(1)
memory usage: 164.3 MB


In [4]:
uber_df.head()

,Date/Time,Lat,Lon,Base
0,2014-04-01 00:11:00,40.7690,-73.9549,B02512
1,2014-04-01 00:17:00,40.7267,-74.0345,B02512
2,2014-04-01 00:21:00,40.7316,-73.9873,B02512
3,2014-04-01 00:28:00,40.7588,-73.9776,B02512
4,2014-04-01 00:33:00,40.7594,-73.9722,B02512


In [5]:
uber_df.describe()

,Date/Time,Lat,Lon
count,4534327,"4,534,327.0000","4,534,327.0000"
mean,2014-07-11 18:50:50.578152,40.7393,-73.9730
min,2014-04-01 00:00:00,39.6569,-74.9290
25%,2014-05-28 15:18:00,40.7211,-73.9965
50%,2014-07-17 14:45:00,40.7422,-73.9834
75%,2014-08-27 21:55:00,40.7610,-73.9653
max,2014-09-30 22:59:00,42.1166,-72.0666
std,NaN,0.0399,0.0573


In [6]:
uber_df.isna().sum()

Date/Time    0
Lat          0
Lon          0
Base         0
dtype: int64

## 1B. Features

In [7]:
# -----------------------------------------------------------------------------
# DATE/TIME PREPROCESSING
# -----------------------------------------------------------------------------
uber_df['date'] = uber_df['Date/Time'].dt.date
uber_df['hour'] = uber_df['Date/Time'].dt.hour
uber_df['day_of_week'] = uber_df['Date/Time'].dt.day_name()
uber_df['day_of_week_num'] = uber_df['Date/Time'].dt.dayofweek
uber_df['month'] = uber_df['Date/Time'].dt.month_name()

In [8]:
uber_df.shape

(4534327, 9)

In [9]:
# -----------------------------------------------------------------------------
# NEW YORK CITY BOUNDARIES
# -----------------------------------------------------------------------------
# NYC boundaries, see: https://s-media.nyc.gov/agencies/dcp/assets/files/pdf/data-tools/bytes/nybb_metadata.pdf

nyc_boundaries = uber_df[
    uber_df['Lat'].between(40.5, 40.9) &
    uber_df['Lon'].between(-74.25, -73.7)
]

outside_nyc = uber_df[~uber_df.index.isin(nyc_boundaries.index)]
print(f'Total rows: {len(uber_df)}')
print(f'Inside NYC box: {len(nyc_boundaries)} ({len(nyc_boundaries)/len(uber_df):.2%})')
print(f'Outside NYC box: {len(outside_nyc)} ({len(outside_nyc)/len(uber_df):.2%})')

outside_nyc[['Lat', 'Lon']].describe()

uber_df = nyc_boundaries # Drop points outside of NYC

Total rows: 4534327
Inside NYC box: 4499916 (99.24%)
Outside NYC box: 34411 (0.76%)


**Out of 4 534 327 pickups (April to September 2014):**

- **99.24% (4 499 916)** fall within NYC's official bounding box (per NYC Dept. of City Planning)
- **0.76% (34 411 rows)** sit well outside, extending as far as ~40 miles into Connecticut and New Jersey

These outliers are dropped as noise before clustering.

In [10]:
# -----------------------------------------------------------------------------
# DISPLAYING THE OUTSIDE NYC DATA ON A MAP
# -----------------------------------------------------------------------------

fig = px.scatter_map(
    outside_nyc, 
    lat='Lat', 
    lon='Lon',
    zoom=7, 
    height=700,
    width=1200,
)

fig.update_traces(marker=dict(color='red'))

fig.update_layout(
    title=dict(
        text=f'Removed Outside NYC Pickups Data ({len(outside_nyc)/len(uber_df):.2%})',
        x=0.5,
        xanchor='center'
    ))

fig.update_layout(margin={'r':0,'t':40,'l':0,'b':0})
fig.show()

**Limitations:** The NYC boundary filter uses a rectangular bounding box (lat 40.5–40.9, lon -74.25 to -73.7) rather than the city's true, irregular shape.

Because a bounding box must be wide enough to include all five boroughs (including Staten Island deep in the south-west), the boundaries also capture areas that are not NYC, most notably parts of New Jersey near Newark Airport.

A more precise fix would use the actual NYC polygon instead of a rectangular box. This was not implemented due to time constraints, and because refining the exact geographic boundary was not the focus of this project.

However it's a reasonable next step for future improvement.

## 1C. Functions

In [11]:
# -----------------------------------------------------------------------------
# FUNCTIONS DEFINITIONS
# -----------------------------------------------------------------------------


# SLICES DATA BY DAY OF THE WEEK AND HOUR
def day_hour_pickups(data: pd.DataFrame, day: str, hour: int) -> pd.DataFrame:
    '''Return pickups for a single hour of a single day of week'''
    return data[(data['day_of_week'] == day) & (data['hour'] == hour)]



# COMPUTE CLUSTERS FOR ALL DAY AND HOUR COMBINATIONS
def clusters_compute(data: pd.DataFrame, days: list[str], k: int = 7, random_state: int = 0) -> tuple[pd.DataFrame, pd.DataFrame]:
    '''
    Runs KMeans clustering for every day of the week and hours combination

    Returns:
        hotzones_summary_df : one row per cluster per day/hour, with center coords and pickup count
        pickups_with_clusters_df    : original pickup-level data, each row tagged with its cluster_id
    '''

    hotzones_summary_df = pd.DataFrame() # clusters for all day and hours combinations
    pickups_with_clusters_df = pd.DataFrame()    # uber_df with a cluster_id column


    for day in days:
        for hour in range(24):
            # Keep only the data we need
            filtered_data = day_hour_pickups(data, day, hour)
            X = filtered_data[['Lat', 'Lon']]

            # Scale the data
            sc = StandardScaler()
            X = sc.fit_transform(X)

            # Train the model
            kmeans = KMeans(n_clusters = k, random_state = random_state, n_init = 'auto')
            kmeans.fit(X)

            # Convert back the scaled coordinates values to real values
            cluster_centers = sc.inverse_transform(kmeans.cluster_centers_)
            cluster_centers_df = (pd.DataFrame(cluster_centers, columns=['cluster_lat', 'cluster_lon'])
                        .reset_index()
                        .rename(columns={'index': 'cluster_id'})
                        )

            # Create a dateframe for all day and hours combinations clusters
            filtered_data['cluster_id'] = kmeans.labels_
            filtered_data_count_df = pd.DataFrame(filtered_data.groupby('cluster_id').size()).reset_index().rename(columns={0: 'count'})
            filtered_data_count_df = filtered_data_count_df.merge(cluster_centers_df, on='cluster_id', how='left')
            filtered_data_count_df['day'] = day
            filtered_data_count_df['hour'] = hour
            hotzones_summary_df = pd.concat([hotzones_summary_df, filtered_data_count_df])

            # Create a new dataframe of our data with cluster_id column
            uber_filtered_data_df = filtered_data.copy()
            uber_filtered_data_df['cluster_id'] = kmeans.labels_
            pickups_with_clusters_df = pd.concat([pickups_with_clusters_df, uber_filtered_data_df]) 

        # Print a message every 24 operations to show progress   
        print(f'✓ {day} combinations computed')

    # return results
    return hotzones_summary_df, pickups_with_clusters_df



# Calculate distances between points
def pickup_distance(data: pd.DataFrame, kmeans: KMeans, cluster_col: str = 'cluster_id') -> pd.DataFrame:
    '''
    For each cluster, compute the mean and max distance (in km) from its
    points to its own KMeans center.
    '''
    results = []
    for c in sorted(data[cluster_col].unique()):
        points = np.radians(data[data[cluster_col] == c][['Lat', 'Lon']].values)
        center = np.radians(kmeans.cluster_centers_[int(c)].reshape(1, -1))  # convert center too
        dists_km = haversine_distances(points, center) * 6371

        results.append({
            'cluster_id': c,
            'mean_km': dists_km.mean(),
            'median_km': np.median(dists_km),
            'max_km': dists_km.max(),
            'n_points': len(points)
        })

    return pd.DataFrame(results)

# 2. Start small

## 2A. Visualizations

In [12]:
# -----------------------------------------------------------------------------
# TOTAL PICKUPS BY HOUR OF THE DAY
# -----------------------------------------------------------------------------

hourly_counts = uber_df.groupby('hour').size()

fig = px.bar(
    hourly_counts,
    x=hourly_counts.index,
    y=hourly_counts.values,
    labels={'x': 'Hour of day', 
            'y': 'Number of pickups'},
    height=400,
    width=1000,
)

fig.update_layout(
    title=dict(
        text='Total pickups by hour of day (april > september 2014)',
        x=0.5,
        xanchor='center'
    ))

fig.show()


# Heatmap of hour vs day-of-week
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

heatmap_df = uber_df.groupby(['day_of_week', 'hour']).size().reset_index(name='count')
heat_pivot = heatmap_df.pivot(index='day_of_week', columns='hour', values='count').reindex(day_order) # To get days in the order defined

fig = px.imshow(
    heat_pivot,
    labels=dict(x='Hour of day', y='Day of week', color='Pickups'),
    color_continuous_scale='YlOrRd',
    height=400,
    width=1000,
)

fig.update_layout(
    title=dict(
        text='Pickup volume heatmap (april > september 2014)',
        x=0.5,
        xanchor='center'
    ))

fig.show()

Across the full April to September 2014 window, pickup volume by day-of-week/hour ranges from **2 549 pickups (Tuesday, 2 AM)** (the quietest slice) up to **56 394 pickups (Thursday, 5 PM)** (the busiest), a >22x difference.

This wide range is the reason hot zones need to be computed per time slice rather than once for the whole dataset: a single city-wide clustering would be dominated by rush-hour demand and say nothing useful about 2 AM.

## 2B. Pick sample

We're going to check the busiest combination: thursday @ 5 pm

In [13]:
# -----------------------------------------------------------------------------
# PICKING ONE SAMPLE TO CHECK KMEANS RESULTS
# -----------------------------------------------------------------------------

sample_df = day_hour_pickups(uber_df, 'Thursday', 17)

display(sample_df.head())
print(f"There's {sample_df.shape[0]} rows in our test sample")

,Date/Time,Lat,Lon,Base,date,hour,day_of_week,day_of_week_num,month
3119,2014-04-03 17:00:00,40.7675,-73.9666,B02512,2014-04-03,17,Thursday,3,April
3120,2014-04-03 17:00:00,40.7688,-73.8624,B02512,2014-04-03,17,Thursday,3,April
3121,2014-04-03 17:01:00,40.7356,-74.0079,B02512,2014-04-03,17,Thursday,3,April
3122,2014-04-03 17:02:00,40.6816,-73.9255,B02512,2014-04-03,17,Thursday,3,April
3123,2014-04-03 17:02:00,40.7677,-73.9826,B02512,2014-04-03,17,Thursday,3,April


There's 56394 rows in our test sample


In [14]:
# -----------------------------------------------------------------------------
# DISPLAYING THE DATA ON A MAP
# -----------------------------------------------------------------------------
fig = px.scatter_map(
    sample_df, 
    lat='Lat', 
    lon='Lon',
    zoom=10, 
    height=700,
    width=1200,
)

fig.update_layout(
    title=dict(
        text='NYC Pickups - Thursday 5 PM',
        x=0.5,
        xanchor='center'
    ))

fig.update_layout(margin={'r':0,'t':40,'l':0,'b':0})
fig.show()

## 2C. Elbow

In [15]:
# -----------------------------------------------------------------------------
# CHECKING ELBOW RESULT
# -----------------------------------------------------------------------------

X = sample_df[['Lat', 'Lon']]

# Initialize StandardScaler
# StandardScaler will substract mean and divide by standard deviation to each observation
sc = StandardScaler()

# Apply StandardScaler to X
X = sc.fit_transform(X)

# Visualize first five rows
# Standard scaler transform X as numpy array. Therefore you can't use .head()
X[:5]


# Let's create a loop that will collect the Within-sum-of-square (wcss) for each value K
# Let's use .inertia_ parameter to get the within sum of square value for each value K
wcss =  []
k = []
for i in range (2, 11):
    kmeans = KMeans(n_clusters = i, random_state = 0, n_init = 'auto')
    kmeans.fit(X)
    wcss.append(kmeans.inertia_)
    k.append(i)
    print('WCSS for K={} --> {}'.format(i, wcss[-1]))



# Create DataFrame
wcss_frame = pd.DataFrame(wcss)
k_frame = pd.Series(k)

# Create figure
fig= px.line(
    wcss_frame,
    x=k_frame,
    y=wcss_frame.iloc[:,-1]
)

# Create title and axis labels
fig.update_layout(
    yaxis_title='Inertia',
    xaxis_title='# Clusters',
    height=400,
    width=1000,
    title=dict(
    text='Inertia per cluster',
    x=0.5,
    xanchor='center' 
    )
)

# Render
#fig.show(renderer='notebook')
fig.show() # if using workspace

WCSS for K=2 --> 78630.54866229993
WCSS for K=3 --> 50969.18382415749
WCSS for K=4 --> 37052.16832807037
WCSS for K=5 --> 31027.97765083552
WCSS for K=6 --> 23341.80305865286
WCSS for K=7 --> 17482.879612337136
WCSS for K=8 --> 16741.918490928958
WCSS for K=9 --> 13860.111220645824
WCSS for K=10 --> 11932.2389071943


Inertia drops sharply through k=2 to 7, then the rate of decrease slows noticeably from k=7 onward. Each additional cluster beyond this point gives comparatively smaller reductions in within-cluster variance, consistent with a visual 'elbow' around **k=7**.

We'll now run the silhouette test to check whether increasing k beyond 7 (to 8, 9, or 10) meaningfully improves cluster quality.

## 2D. Silhouette

In [16]:
# -----------------------------------------------------------------------------
# COMPUTING SILHOUETTE
# -----------------------------------------------------------------------------

# Computer mean silhouette score
sil = []
k = []

## Careful, you need to start at i=2 as silhouette score cannot accept less than 2 labels
for i in range (2, 11):
    kmeans = KMeans(n_clusters = i, random_state = 0, n_init = 'auto')
    kmeans.fit(X)
    sil.append(silhouette_score(X, kmeans.predict(X)))
    k.append(i)
    print('Silhouette score for K={} is {}'.format(i, sil[-1]))


# Create a data frame
cluster_scores=pd.DataFrame(sil)
k_frame = pd.Series(k)

# Create figure
fig = px.bar(data_frame=cluster_scores,
             x=k,
             y=cluster_scores.iloc[:, -1]
            )

# Add title and axis labels
fig.update_layout(
    yaxis_title='Silhouette Score',
    xaxis_title='# Clusters',
    height=400,
    width=1000,
    title=dict(
        text='Silhouette Score per cluster',
        x=0.5,
        xanchor='center' 
        )
)

# Render
#fig.show(renderer='notebook')
fig.show() # if using workspace

Silhouette score for K=2 is 0.6854034440288592
Silhouette score for K=3 is 0.4407784412339878
Silhouette score for K=4 is 0.5006811934146728
Silhouette score for K=5 is 0.36634166870247853
Silhouette score for K=6 is 0.3878687346750875
Silhouette score for K=7 is 0.4705936253206298
Silhouette score for K=8 is 0.4655422057448663
Silhouette score for K=9 is 0.42538309914986155
Silhouette score for K=10 is 0.42793116383959184


Among higher values, **k=7 (0.471)** is the best-performing choice within the 7 to 10 range checked here, edging out k=8 (0.466) and clearly outperforming k=9 to 10 (~0.42 to 0.43). Combined with the elbow bend around the same point, **k=7** is the most defensible choice: a good balance of cluster quality and granularity.

## 2E. Map result

In [17]:
# -----------------------------------------------------------------------------
# DOING FULL COMPUTATION WITH OUR SELECTED CLUSTER NUMBER
# -----------------------------------------------------------------------------

k_val = 7

kmeans = KMeans(n_clusters = k_val, random_state = 0, n_init = 'auto')
kmeans.fit(X)

cluster_centers = sc.inverse_transform(kmeans.cluster_centers_)
cluster_centers_df = (pd.DataFrame(cluster_centers, columns=['cluster_lat', 'cluster_lon'])
                      .reset_index()
                      .rename(columns={'index': 'cluster_id'})
                    )

sample_df['cluster_id'] = kmeans.labels_
sample_count_df = pd.DataFrame(sample_df.groupby('cluster_id').size()).reset_index().rename(columns={0: 'count'})
sample_count_df = sample_count_df.merge(cluster_centers_df, on='cluster_id', how='left')


fig = px.scatter_map(
    sample_df, 
    lat='Lat', 
    lon='Lon',
    zoom=10,
    color='cluster_id',
    height=700,
    width=1200,
    title='NYC Pickups - Thursday 5 PM'
)

# Add cluster centers on top
fig.add_trace(go.Scattermap(
    lat=cluster_centers[:, 0],
    lon=cluster_centers[:, 1],
    mode='markers+text',
    marker=dict(size=12, color='black', symbol='circle'),
    text=[f'Cluster #{i}' for i in range(len(cluster_centers))],
    textposition='top right',
    name='Cluster centers',
    
))

fig.update_layout(
    margin={'r':0,'t':40,'l':0,'b':0},
    title=dict(
        text='NYC Pickups - Thursday 5 PM',
        x=0.5,
        xanchor='center'
    ))
fig.show()

## 2F. Business logic

In [18]:
X_deg = sample_df[['Lat', 'Lon']].values
kmeans_deg = KMeans(n_clusters = k_val, random_state = 0, n_init = 'auto').fit(X_deg)
sample_df['cluster_id'] = kmeans_deg.labels_

pickup_distance(sample_df, kmeans_deg, 'cluster_id')

,cluster_id,mean_km,median_km,max_km,n_points
0,0,2.1872,1.6579,15.5990,16170
1,1,1.2241,1.1793,16.3659,29316
2,2,4.4220,4.1489,13.2372,189
3,3,1.1852,0.6122,10.9512,862
4,4,1.7911,1.4720,18.1205,7425
5,5,2.0289,1.2618,21.0704,406
6,6,2.0860,0.9905,15.3034,2026


Uber's rider tolerance threshold translates to distance requirements:

- **Wait time limit:** 5-7 minutes (after which riders start canceling)
- **Typical NYC traffic speed:** ~15 km/h in dense urban areas
- **Distance covered:** 1.25-1.75 km in 5-7 minutes at this speed

This means for hot zones to actually meet Uber's wait-time promise, the average distance from a cluster's pickups to its center should stay within that **~1.25-1.75 km range**.


**Manhattan clusters (1, 4, 6):**

- **Cluster 1** (median 1.18 km): Below mean value
- **Cluster 4** (median 1.47 km): Slightly higher but acceptable
- **Cluster 6** (median 0.99 km): Below target median, higher mean but covers Bronx too, where faster speeds apply

**Peripheral clusters (0, 3, 5):** Larger medians (0.61-1.66 km), but faster travel outside Manhattan makes them reasonable.

**Cluster 2 outlier:** Median 4.15 km, mean 4.42 km. At 50 km/h (plausible for peripheral location), 5-7 minutes = 4.17 to 5.83 km range. Both values fall within reach.

**Bottom line:** Dense Manhattan zones stay tight for slow traffic, peripheral zones cover more distance but allow faster speeds.

**Conclusion:** Our clustering is well-supported both methodologically (validated via elbow and silhouette) and against the business constraint. Median and mean pickup-to-center distances fall within a reasonable range for most clusters.

# 3. Generalization

Ideally, we would re-run the full elbow + silhouette search independently for each of the 168 day-of-week × hour combinations, letting each slice pick its own optimal **k**. In practice for this project, this is not computationally reasonable: silhouette score takes a lot of time to compute. Repeating that search 168 times (each on slices ranging from ~2 500 to ~56 000 points) would take hours for limited benefit.

Instead, we use **k=7**, validated on the busiest slice (Thursday, 5 PM) via elbow and silhouette, as a fixed baseline applied uniformly across all 168 combinations. This is a deliberate simplification. A possible refinement for future work would be optimizing **k** per slice, rather than one global value.

## 3A. Checks

In [19]:
# -----------------------------------------------------------------------------
# CHECKING VALUES
# -----------------------------------------------------------------------------

biggest_value = heatmap_df[heatmap_df['count'] == heatmap_df['count'].max()]
lowest_value = heatmap_df[heatmap_df['count'] == heatmap_df['count'].min()]

print(f"Busiest: {biggest_value['day_of_week'].iloc[0]} @ {biggest_value['hour'].iloc[0]}:00 - {biggest_value['count'].iloc[0]} pickups")
print(f"Quietest: {lowest_value['day_of_week'].iloc[0]} @ {lowest_value['hour'].iloc[0]}:00 - {lowest_value['count'].iloc[0]} pickups")

Busiest: Thursday @ 17:00 - 56394 pickups
Quietest: Tuesday @ 2:00 - 2549 pickups


This confirms Thursday, 5 PM as the busiest slice in the dataset (56 394 pickups) - matching 
the sample used for the k-selection above - and Tuesday, 2 AM as the quietest (2 549 pickups). 
Because even the quietest slice has 2 549+ points, every one of the 168 day/hour combinations 
has enough data to support a meaningful k=7 clustering; no slice needs a reduced cluster count.

## 3B. Computing

In [20]:
# -----------------------------------------------------------------------------
# COMPUTING ALL DAY x HOUR COMBINATIONS
# -----------------------------------------------------------------------------
hotzones_summary_df, pickups_with_clusters_df = clusters_compute(uber_df, day_order, k = k_val)

# Converting cluster_id to string to avoid gradient scale color
# Otherwise some clusters are hard to see
hotzones_summary_df['cluster_id'] = hotzones_summary_df['cluster_id'].astype(str)
pickups_with_clusters_df['cluster_id'] = pickups_with_clusters_df['cluster_id'].astype(str)

✓ Monday combinations computed
✓ Tuesday combinations computed
✓ Wednesday combinations computed
✓ Thursday combinations computed
✓ Friday combinations computed
✓ Saturday combinations computed
✓ Sunday combinations computed


Maps visualization are available in the uber dashboard and won't be computed in this notebook to keep it as clean as possible.

## 3C. Maps

Per-hour maps for each day (168 combinations total) are generated separately and served in our dashboard rather than rendered in this notebook. Displaying 24 full interactive maps in a single Jupyter/VS Code cell is leading to crashs and is not useful for reviewing results anyway.

The Thursday, 5 PM map in section 2 already demonstrates the clustering and mapping approach on a representative slice. The same logic applies uniformly across all day/hour combinations via `hotzones_summary_df`.

In [21]:
# Disabled: Rendering 24 maps in one cell crashes VS Code/Jupyter. See note above for details.
# # -----------------------------------------------------------------------------
# # MAPS
# # -----------------------------------------------------------------------------
# for hour in range(24):
#     fig = px.scatter_map(
#         hotzones_summary_df[(hotzones_summary_df['day'] == 'Monday') & (hotzones_summary_df['hour'] == hour)], 
#         lat='cluster_lat', 
#         lon='cluster_lon',
#         size='count',
#         zoom=10,
#         color='cluster_id',
#         color_discrete_sequence=px.colors.qualitative.Bold,
#         height=700,
#         width=1200,
#     )

#     fig.update_layout(
#     margin={'r':0,'t':40,'l':0,'b':0},
#     title=dict(
#         text=f'NYC Pickups - Monday {hour}:00',
#         x=0.5,
#         xanchor='center'
#     ))

#     fig.show()

# 4. Cross-checking dataset

The clustering above is built entirely from 2014 GPS coordinates. As a sanity check, we use a 
second, independent 2015 dataset - pickups tagged by TLC zone ID rather than raw coordinates - 
and rank zones by pickup volume for the exact same slice (Thursday, 5 PM), using a completely 
different method (frequency counts, not clustering). If the two agree, that's strong evidence 
the hot zones reflect real demand patterns rather than an artifact of one method or one year.

In [22]:
taxi_zones = pd.read_csv(f'{path}{taxi_file}')
display(taxi_zones.head())

uber_2015_df = pd.read_csv(f'{path}{raw_2015}', parse_dates=['Pickup_date'], date_format='%Y-%m-%d %H:%M:%S')
display(uber_2015_df.head())

,LocationID,Borough,Zone
0,1,EWR,Newark Airport
1,2,Queens,Jamaica Bay
2,3,Bronx,Allerton/Pelham Gardens
3,4,Manhattan,Alphabet City
4,5,Staten Island,Arden Heights


,Dispatching_base_num,Pickup_date,Affiliated_base_num,locationID
0,B02617,2015-05-17 09:47:00,B02617,141
1,B02617,2015-05-17 09:47:00,B02617,65
2,B02617,2015-05-17 09:47:00,B02617,100
3,B02617,2015-05-17 09:47:00,B02774,80
4,B02617,2015-05-17 09:47:00,B02617,90


In [23]:
print(f'Pickup locations reference:')
print()
taxi_zones.info()
print('\n\n')
print(f'Pickup zones data:')
print()
uber_2015_df.info(show_counts=True)

Pickup locations reference:

<class 'pandas.DataFrame'>
RangeIndex: 265 entries, 0 to 264
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   LocationID  265 non-null    int64
 1   Borough     265 non-null    str  
 2   Zone        265 non-null    str  
dtypes: int64(1), str(2)
memory usage: 12.4 KB



Pickup zones data:

<class 'pandas.DataFrame'>
RangeIndex: 14270479 entries, 0 to 14270478
Data columns (total 4 columns):
 #   Column                Non-Null Count     Dtype         
---  ------                --------------     -----         
 0   Dispatching_base_num  14270479 non-null  str           
 1   Pickup_date           14270479 non-null  datetime64[us]
 2   Affiliated_base_num   14108284 non-null  str           
 3   locationID            14270479 non-null  int64         
dtypes: datetime64[us](1), int64(1), str(2)
memory usage: 599.6 MB


In [24]:
# -----------------------------------------------------------------------------
# DATE/TIME PREPROCESSING
# -----------------------------------------------------------------------------

uber_2015_df["hour"] = uber_2015_df["Pickup_date"].dt.hour
uber_2015_df["day_of_week"] = uber_2015_df["Pickup_date"].dt.day_name()

In [25]:
# -----------------------------------------------------------------------------
# MERGING TAXIS ZONES
# -----------------------------------------------------------------------------

uber_2015_df = (uber_2015_df.merge(taxi_zones, 
                                left_on="locationID", 
                                right_on="LocationID", 
                                how="left")
                                .drop(columns=['Dispatching_base_num', 'Affiliated_base_num', 'locationID', 'LocationID']))

display(uber_2015_df.head())
print(uber_2015_df['Borough'].isna().sum(), "rows with no matching zone")

,Pickup_date,hour,day_of_week,Borough,Zone
0,2015-05-17 09:47:00,9,Sunday,Manhattan,Lenox Hill West
1,2015-05-17 09:47:00,9,Sunday,Brooklyn,Downtown Brooklyn/MetroTech
2,2015-05-17 09:47:00,9,Sunday,Manhattan,Garment District
3,2015-05-17 09:47:00,9,Sunday,Brooklyn,East Williamsburg
4,2015-05-17 09:47:00,9,Sunday,Manhattan,Flatiron


0 rows with no matching zone


In [26]:
# -----------------------------------------------------------------------------
# DISPLAY TAXIS ZONES
# -----------------------------------------------------------------------------

zone_counts_df = (
    uber_2015_df[(uber_2015_df['day_of_week'] == 'Thursday') & (uber_2015_df['hour'] == 17)]
    .groupby(['Borough', 'Zone'])
    .size()
    .reset_index(name='pickup_count')
    .sort_values('pickup_count', ascending=False)
)

fig = px.bar(
    zone_counts_df.head(40),
    x='pickup_count', 
    y='Zone', 
    color='Borough',
    orientation='h',
    height=800,
    width=1200,
)
fig.update_xaxes(title_text='Pickup count')
fig.update_yaxes(title_text='Zones')
fig.update_layout(
    yaxis={'categoryorder': 'total ascending'},
    title=dict(
        text='NYC Pickups - Thursday 5 PM | 2015',
        x=0.5,
        xanchor='center'
    )
)

fig.show()

**Top 2015 zones for Thursday, 5 PM:** Almost entirely Manhattan, led by Midtown Center (7 479 pickups), Union Sq (5 459), and Midtown East (4 955), with LaGuardia Airport (3 698) and JFK Airport (2 083) as the two standout non-Manhattan hot spots.

**Alignment with 2014 KMeans centers:**

- **Cluster 4 (40.754, -73.982):** Sits squarely in Midtown Manhattan, matching the #1 2015 zone (Midtown Center) almost exactly
- **Cluster 3 (40.768, -73.868):** Lands near LaGuardia Airport
- **Cluster 2 (40.651, -73.786):** lands directly on JFK. Both 2014 clusters correctly isolate the two airports as distinct hot zones, matching their standalone ranking in 2015 data
- **Cluster 6 (40.784, -73.960):** Covers the Upper East Side, consistent with that zone's strong 2015 showing (Upper East Side South: 4,184 pickups)

**Conclusions:** Two different years, two different data formats (coordinates vs. zone IDs), and two different methods (geometric clustering vs. frequency ranking) converge on the same physical locations: Midtown, the Upper East Side, and both airports as Thursday-evening hot zones. This agreement is the strongest evidence in this project that the clustering approach generalizes.

In [27]:
# -----------------------------------------------------------------------------
# HEATMAP OF HOURS vs DAY OF THE WEEK BY BOROUGH
# -----------------------------------------------------------------------------

borough_heatmap_df = (
    uber_2015_df
    .groupby(['Borough', 'day_of_week', 'hour'])
    .size()
    .reset_index(name='count')
)


fig = px.density_heatmap(
    borough_heatmap_df,
    x='hour',
    y='day_of_week',
    z='count',
    facet_col='Borough',
    facet_col_wrap=7,
    category_orders={'day_of_week': day_order},
    color_continuous_scale='Plasma',
    height=300,
    width=1500,
    # title='Pickup volume by borough, day of week, and hour (2015 Jan-Jun data)'
)

fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1])) # Remove Borough= in front of every neighborhood
fig.update_xaxes(title_text='')
fig.update_yaxes(title_text='')
fig.update_yaxes(title_text='Day of week', col=1)
fig.update_xaxes(title_text='Hour', col=4)
fig.update_layout(
    title=dict(
        text='Pickup volume by borough, day of week, and hour (January-June 2015)',
        x=0.5,
    ))
fig.show()

In [28]:
borough_totals = (pd.DataFrame(
    borough_heatmap_df.groupby('Borough')['count'].sum()
    .sort_values(ascending=False))
    .reset_index())

borough_totals['percentage'] = (borough_totals['count'] / borough_totals['count'].sum()).mul(100).round(2).apply(lambda x: f'{x:.2f}%')
display(borough_totals)

,Borough,count,percentage
0,Manhattan,10371060,72.67%
1,Brooklyn,2322000,16.27%
2,Queens,1343945,9.42%
3,Bronx,220146,1.54%
4,Staten Island,6959,0.05%
5,Unknown,6264,0.04%
6,EWR,105,0.00%


**Borough-level demand is extremely concentrated:** Manhattan alone accounts for 72.7% of all 2015 pickups, with Brooklyn (16.3%) and Queens (9.4%) making up most of the remainder. Together, these three boroughs cover 98.4% of demand. The Bronx (1.5%), Staten Island, and Newark Airport (EWR) are negligible by comparison.

This mirrors the density pattern found in the 2014 clustering.

# 5. Exports

In [32]:
# -----------------------------------------------------------------------------
# DATAFRAMES EXPORTS
# -----------------------------------------------------------------------------
dataframes_list = [hotzones_summary_df, pickups_with_clusters_df.drop(columns=['Date/Time', 'Base', 'day_of_week_num'])]
file_names = ['hotzones_summary', 'pickups_with_clusters']

for dataframe, file in zip(dataframes_list, file_names):
    dataframe.to_csv(f'{file}.csv', index=False)
    print(f'✓ Saved {file}.csv')

✓ Saved hotzones_summary.csv
✓ Saved pickups_with_clusters.csv


# Conclusion

- **Demand varies more than 20x across the week:** 2,549 to 56,394 pickups per hour-slot, confirming hot zones must be computed per day/hour rather than city-wide
- **k=7 clusters:** Chosen via elbow + silhouette on the busiest slice, has enough data to apply across all 168 day/hour combinations. However, a distance-to-center check reveals a limitation: while median/mean pickup-to-center distances are mostly within Uber's implied wait-time radius, every cluster's *max* distance badly exceeds it
- **Cross-validation success:** Independent 2015 dataset confirms the 2014 KMeans hot zones land on real, high-traffic locations (Midtown, Upper East Side, JFK, LaGuardia) rather than being an artifact of the method or the year